# Call of Acaristan — Qwen3-TTS 1.7B Voice Generator (Free Colab T4)

1. Click **Runtime -> Change runtime type -> T4 GPU -> Save**.
2. Click **Runtime -> Run all**.
3. When Cell 2 asks, upload **`build/vo_colab_kit.zip`**.
4. Wait ~15–20 minutes — Cell 5 downloads **`vo_generated.zip`** containing all missing `VO_xxx.wav` files.
5. Back on your PC, run:
   ```bash
   python tools/vo_batch.py colab-import "C:\Users\YOUR_USER\Downloads\vo_generated.zip"
   python tools/build_radio_cache.py
   python tools/voice_audit.py --check
   ```

In [ ]:
# Cell 1: Verify GPU and install qwen-tts
import torch
assert torch.cuda.is_available(), (
    'No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU -> Save, then re-run.'
)
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}')
!pip install -q qwen-tts soundfile

In [ ]:
# Cell 2: Upload and unpack vo_colab_kit.zip
import json, os, zipfile
from pathlib import Path
from google.colab import files

KIT_ZIP = Path('/content/vo_colab_kit.zip')
KIT_DIR = Path('/content/vo_kit')
OUT_DIR = Path('/content/vo_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not KIT_ZIP.exists():
    print('Select build/vo_colab_kit.zip from your PC:')
    up = files.upload()
    first = next(iter(up.keys()))
    if Path(first).resolve() != KIT_ZIP.resolve():
        Path(first).rename(KIT_ZIP)

with zipfile.ZipFile(KIT_ZIP, 'r') as zf:
    zf.extractall(KIT_DIR)

MANIFEST = json.loads((KIT_DIR / 'jobs.json').read_text(encoding='utf-8'))
GROUPS = MANIFEST['groups']
JOBS = MANIFEST['jobs']
n_clone = sum(1 for g in GROUPS.values() if g['mode'] == 'clone_existing')
n_design = sum(1 for g in GROUPS.values() if g['mode'] == 'design')
print(f'Loaded {len(JOBS)} missing lines across {len(GROUPS)} cast groups '
      f'({n_clone} existing voice clones, {n_design} VoiceDesign characters).')

In [ ]:
# Cell 3: Phase 1 — Design anchor voices for brand-new characters (Maru, Retardovici, Ucilev, Alex, etc.)
import gc
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

def load_qwen(model_id, dtype):
    try:
        return Qwen3TTSModel.from_pretrained(
            model_id, device_map='cuda:0', dtype=dtype, attn_implementation='sdpa'
        )
    except TypeError:
        return Qwen3TTSModel.from_pretrained(
            model_id, device_map='cuda:0', torch_dtype=dtype, attn_implementation='sdpa'
        )

design_groups = [g for g in GROUPS.values() if g['mode'] == 'design']
ref_dir = KIT_DIR / 'refs'
ref_dir.mkdir(parents=True, exist_ok=True)

need_design = [g for g in design_groups if not (ref_dir / f"{g['group_id']}.wav").exists()]
if need_design:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f'Loading Qwen3-TTS-12Hz-1.7B-VoiceDesign ({dtype})...')
    design_model = load_qwen('Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', dtype)
    for g in need_design:
        gid = g['group_id']
        anchor_wav = ref_dir / f'{gid}.wav'
        print(f'Designing voice for {gid} ({g["folder"]})...')
        for attempt in range(3):
            try:
                wavs, sr = design_model.generate_voice_design(
                    text=g['anchor_text'],
                    language='English',
                    instruct=g['instruct'],
                )
                wav = np.asarray(wavs[0], dtype=np.float32)
                if len(wav) / sr >= 1.5 and float(np.max(np.abs(wav))) > 0.01:
                    sf.write(str(anchor_wav), wav, sr, subtype='PCM_16')
                    break
            except RuntimeError as exc:
                print(f'  retry {attempt + 1} ({exc})')
                if 'probability' in str(exc).lower() and dtype == torch.float16:
                    del design_model; gc.collect(); torch.cuda.empty_cache()
                    dtype = torch.float32
                    design_model = load_qwen('Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', dtype)
        assert anchor_wav.exists(), f'Failed to design anchor for {gid}'
        display(Audio(str(anchor_wav)))
    del design_model
    gc.collect()
    torch.cuda.empty_cache()
    print('Phase 1 complete — VoiceDesign model unloaded to free VRAM.')
else:
    print('All VoiceDesign anchors already exist (delete any refs/<ID>.wav to re-roll a voice).')

In [ ]:
# Cell 4: Phase 2 — Clone every character (existing cast + designed anchors) for all missing lines
import gc, time, zipfile
import numpy as np
import soundfile as sf
import torch
from qwen_tts import Qwen3TTSModel

def _save_backup_zip():
    zpath = Path('/content/vo_generated.zip')
    with zipfile.ZipFile(zpath, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(OUT_DIR.rglob('*.wav')):
            zf.write(p, p.relative_to(OUT_DIR).as_posix())
    return zpath

base_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Loading Qwen3-TTS-12Hz-1.7B-Base ({base_dtype})...')
base_model = load_qwen('Qwen/Qwen3-TTS-12Hz-1.7B-Base', base_dtype)

by_group = {}
for job in JOBS:
    by_group.setdefault(job['group_id'], []).append(job)

t_start = time.time()
done = 0
failed = []
total = len(JOBS)

for gid in sorted(by_group):
    g = GROUPS[gid]
    g_jobs = by_group[gid]
    pending = [
        j for j in g_jobs
        if not (OUT_DIR / j['folder'] / f"{j['vo_id']}.wav").exists()
    ]
    done += len(g_jobs) - len(pending)
    if not pending:
        continue

    ref_wav = str(KIT_DIR / (g['ref_wav'] if g['mode'] == 'clone_existing' else f'refs/{gid}.wav'))
    ref_text = g['ref_text'] if g['mode'] == 'clone_existing' else g['anchor_text']
    print(f'\n=== {gid} ({len(pending)} lines) ===')
    prompt = base_model.create_voice_clone_prompt(ref_audio=ref_wav, ref_text=ref_text)

    for job in pending:
        vo_id = job['vo_id']
        out_wav = OUT_DIR / job['folder'] / f'{vo_id}.wav'
        out_wav.parent.mkdir(parents=True, exist_ok=True)
        exp_s = max(0.7, job['units'] * 0.068)
        ok = False
        for attempt in range(3):
            try:
                wavs, sr = base_model.generate_voice_clone(
                    text=job['tts_text'],
                    language='English',
                    voice_clone_prompt=prompt,
                )
                wav = np.asarray(wavs[0], dtype=np.float32)
                dur = len(wav) / float(sr)
                pk = float(np.max(np.abs(wav))) if wav.size else 0.0
                if pk > 0.01 and 0.20 <= dur <= max(18.0, exp_s * 3.8):
                    sf.write(str(out_wav), wav, sr, subtype='PCM_16')
                    done += 1
                    elapsed = (time.time() - t_start) / 60.0
                    print(f'  [{done:3d}/{total}] {vo_id} ({dur:4.2f}s) [{elapsed:4.1f}m] {job["tts_text"][:58]}')
                    ok = True
                    break
                print(f'  [retry {attempt + 1}] {vo_id}: bad dur={dur:.2f}s pk={pk:.3f}')
            except RuntimeError as exc:
                msg = str(exc).lower()
                print(f'  [error {attempt + 1}] {vo_id}: {exc}')
                if ('probability' in msg or 'nan' in msg or 'inf' in msg) and base_dtype == torch.float16:
                    print('  -> Switching Base model to float32 on T4 for numerical stability...')
                    del base_model, prompt; gc.collect(); torch.cuda.empty_cache()
                    base_dtype = torch.float32
                    base_model = load_qwen('Qwen/Qwen3-TTS-12Hz-1.7B-Base', base_dtype)
                    prompt = base_model.create_voice_clone_prompt(ref_audio=ref_wav, ref_text=ref_text)
        if not ok:
            failed.append(vo_id)
        if done % 50 == 0 and done > 0:
            _save_backup_zip()

zpath = _save_backup_zip()
print(f'\nDONE: {done}/{total} lines generated ({len(failed)} failed). Saved {zpath}')
if failed:
    print('Failed IDs (re-run Cell 4 to retry only these):', failed)

In [ ]:
# Cell 5: Download vo_generated.zip to your PC
from pathlib import Path
from google.colab import files

zpath = Path('/content/vo_generated.zip')
assert zpath.exists(), 'Run Cell 4 first!'
print(f'Downloading {zpath.name} ({zpath.stat().st_size / 1e6:.1f} MB)...')
files.download(str(zpath))